# Cell Subtyping (II) Map My Cells

This workflow uses MapMyCells for cell subtyping.

Input file used here was produced in 'FilteredDataset_03_preprocessing_for_mapmycells.ipyb'

**Note: code adapted from script 3 of ASAP CRN study case tutorial. 

## 1. Package Imports and Configuration

In [ ]:
# Core scientific computing and visualization libraries
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, confusion_matrix

# Standard library imports
import sys
import subprocess
import importlib
import warnings
import os
from pathlib import Path

# Optional: enable cell-level timing for performance awareness
try:
    %load_ext autotime
except ModuleNotFoundError:
    %pip install ipython-autotime
    %load_ext autotime


## 2. Data structure

Define paths and input files

In [ ]:
#set general folder paths
HOME = Path.home()
WS_ROOT = HOME / "workspace"
DATA_DIR = WS_ROOT / "Data"
WS_FILES = WS_ROOT / "ws_files"

if not WS_ROOT.exists():
    print(f"{WS_ROOT} doesn't exist. We need to remount our resources")
    !wb resource mount    

print("Home directory:     ", HOME)
print("Workspace root:     ", WS_ROOT)
print("Data directory:     ", DATA_DIR)
print("ws_files directory: ", WS_FILES)

print("\nContents of workspace root:")
for p in WS_FILES.glob("*"):
    print(" -", p.name, "/" if p.is_dir() else "")

#### Folder to process:
Modify the object below accordingly.

In [ ]:
# Brain region folder to process
region = "substantia_nigra"


In [ ]:

# Define a local path for workshop files
local_data_path = WS_FILES / region

# map my cells directories
mapmycells_input_dir = ( local_data_path / "mapmycells/input" )
mapmycells_output_dir = ( local_data_path / "mapmycells/output" )

# other directories
input_path =( local_data_path / "input")
output_path =( local_data_path / "output")

# Make sure the directories exists
os.makedirs(mapmycells_input_dir, exist_ok=True)
os.makedirs(mapmycells_output_dir, exist_ok=True)
os.makedirs(input_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)

# Create the directory if it doesn't already exist
if not local_data_path.exists():
    print("Local folder missing")

print(f"Local data directory ready at: {local_data_path}")

(info from script 3 of ASAP CRN study case tutorial)

## 3. Reference-Based Mapping with MapMyCells

[MapMyCells](https://portal.brain-map.org/atlases-and-data/bkp/mapmycells) performs **direct taxonomy mapping** of Substantia Nigra–derived cells to a Basal Ganglia reference. Rather than integrating datasets (as in batch-correction approaches like scVI or Harmony), it uses **marker-gene correlation** to assign each cell a taxonomy label and a confidence score.


### 3.1 Set Up MapMyCells Dependencies

We import the `cell_type_mapper` package (which implements the MapMyCells algorithm) and the `abc_atlas_access` package (which provides utilities for downloading reference taxonomy files from the Allen Brain Atlas).

In [ ]:
import json
import cell_type_mapper
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache
from cell_type_mapper.cli.from_specified_markers import FromSpecifiedMarkersRunner

### 3.2 Download Reference Taxonomy Files

MapMyCells requires two precomputed reference files derived from the Basal Ganglia taxonomy:

1. **Precomputed statistics** (`.h5`) — reference cluster expression profiles used for correlation-based assignment  
2. **Query markers** (`.json`) — the set of marker genes used to compare query cells against the reference  

These files are downloaded once and reused across mapping runs.

In [ ]:
# Define paths for MapMyCells reference inputs
precomputed_stats_filepath = (
    mapmycells_input_dir / "Human.precomputed_stats.20250507.h5"
)
query_markers_filepath = (
    mapmycells_input_dir / "Human.query_markers.20250507.json"
)

# Download Basal Ganglia reference taxonomy files (Allen Institute HMBA-BG)
! wget "https://released-taxonomies-802451596237-us-west-2.s3.us-west-2.amazonaws.com/HMBA/BasalGanglia/BICAN_05072025_pre-print_release/MapMyCells/Human.precomputed_stats.20250507.h5" \
    -O {precomputed_stats_filepath}

! wget "https://released-taxonomies-802451596237-us-west-2.s3.us-west-2.amazonaws.com/HMBA/BasalGanglia/BICAN_05072025_pre-print_release/MapMyCells/Human.query_markers.20250507.json" \
    -O {query_markers_filepath}


### 3.3 Configure MapMyCells

Before running MapMyCells, we configure its settings. Key parameters include:

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `normalization` | `"raw"` | Tells MapMyCells our input contains raw counts (normalization is handled internally) |
| `n_processors` | `4` | Number of parallel workers |
| `bootstrap_factor` | `0.5` | Fraction of marker genes resampled per bootstrap iteration |
| `bootstrap_iteration` | `100` | Number of bootstrap rounds to estimate assignment stability |

We also limit thread counts for numerical libraries to avoid oversubscription on shared compute environments.

In [ ]:
import os
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"


# paths to files where mapping output will be written
json_dst_path = str( mapmycells_output_dir / f"human_{region}_mapping.json") 
csv_dst_path = str( mapmycells_output_dir / f"human_{region}_mapping.csv")

processed_filename = (
    input_path / f"asap-cohort_{region}_cells_preprocessed.h5ad"
)

config = {
    "query_path": str(processed_filename),
    "extended_result_path": json_dst_path,
    "csv_result_path": csv_dst_path,
    "verbose_csv": True,
    "query_markers": {
       "serialized_lookup":str(query_markers_filepath)
    },
    "precomputed_stats": {
        "path": str(precomputed_stats_filepath)
    },
    "type_assignment": {
        "n_processors": 4,
        "normalization": "raw",
        "bootstrap_factor": 0.5,
        "bootstrap_iteration": 100
    }
}

### 3.4 Run MapMyCells

In [ ]:
# run map my cells
runner = FromSpecifiedMarkersRunner(
    args=[],
    input_data=config
)
runner.run()

## 4. Analyze and Integrate MapMyCells Results

In this section, we load, standardize, and evaluate the MapMyCells (MMC) outputs.

The goal is to transform raw MMC outputs into:
- **Interpretable cell type annotations** — simplified labels like "Dopaminergic", "GABAergic", etc.  
- **Consistent fields** — standardized column names across different reference workflows  
- **Analysis-ready data** — merged into the AnnData object for visualization and downstream use  

This step bridges **raw mapping output → usable biological annotations**.

### 4.1 Load MapMyCells Results

The MapMyCells CSV output contains one row per cell with:
- **Predicted labels** at each taxonomy level (`class_name`, `subclass_name`, `supertype_name`)  
- **Correlation coefficients** — how similar the cell's expression is to the assigned reference profile  
- **Bootstrapping probabilities** — how consistently the cell maps to the same label across resampled gene sets  

In [ ]:
mmc_output = pd.read_csv(csv_dst_path, comment='#')
mmc_output.columns

### 4.2 Reformat MapMyCells Results
The raw MMC outputs contain detailed but workflow-specific fields (e.g., `class_name`, `subclass_name`, correlation metrics).

We reformat these results to:

- create consistent, intuitive fields across workflows
- derive a simplified cell_type label
- extract key metrics for interpretation:
    - `rho` (correlation strength)
    - `prob` (bootstrapping confidence)

This function performs four tasks:

1. groups detailed MapMyCells labels into broader categories
2. selects the relevant confidence metrics
3. assigns a simplified `cell_type`
4. flags low-confidence assignments as `Unknown`

In [ ]:
def summarize_mmc_results(mmc_results: pd.DataFrame, workflow_name: str):
    """
    Summarize MMC results for human, mouse, or basal ganglia workflows.

    Parameters
    ----------
    mmc_results : pd.DataFrame
        Input dataframe containing MMC outputs.
    workflow_name : str
        One of: "pmdbs_sc_rnaseq", "mouse_sc_rnaseq", "basal_ganglia".

    Returns
    -------
    pd.DataFrame
        Summary dataframe with standardized fields.
    """

    # ------------------------------------------------------------
    # pmdbs_sc_rnaseq
    # ------------------------------------------------------------
    if workflow_name == "pmdbs_sc_rnaseq":
        gabaergic = mmc_results["Class_name"] == "Neuronal: GABAergic"
        glutamatergic = mmc_results["Class_name"] == "Neuronal: Glutamatergic"
        non_neuronal = mmc_results["Class_name"] == "Non-neuronal and Non-neural"

        mmc_results.loc[gabaergic, "phenotype"] = "GABAergic"
        mmc_results.loc[glutamatergic, "phenotype"] = "Glutamatergic"
        mmc_results.loc[non_neuronal, "phenotype"] = mmc_results.loc[
            non_neuronal, "Subclass_name"
        ]

        mmc_results.loc[glutamatergic, "rho"] = mmc_results.loc[
            glutamatergic, "Class_correlation_coefficient"
        ]
        mmc_results.loc[gabaergic, "rho"] = mmc_results.loc[
            gabaergic, "Class_correlation_coefficient"
        ]
        mmc_results.loc[non_neuronal, "rho"] = mmc_results.loc[
            non_neuronal, "Subclass_correlation_coefficient"
        ]

        mmc_results.loc[glutamatergic, "prob"] = mmc_results.loc[
            glutamatergic, "Class_bootstrapping_probability"
        ]
        mmc_results.loc[gabaergic, "prob"] = mmc_results.loc[
            gabaergic, "Class_bootstrapping_probability"
        ]
        mmc_results.loc[non_neuronal, "prob"] = mmc_results.loc[
            non_neuronal, "Subclass_bootstrapping_probability"
        ]

        mmc_results["cell_type"] = mmc_results["phenotype"]

        # Confidence filtering
        mmc_results.loc[mmc_results["rho"] < 0.5, "cell_type"] = "Unknown"
        mmc_results.loc[mmc_results["prob"] < 0.5, "cell_type"] = "Unknown"

        return mmc_results[
            [
                "cell_id"
                "cell_type",
                "phenotype",
                "rho",
                "prob",
                "class_name",
                "subclass_name",
                "supertype_name",
            ]
        ]

    # ------------------------------------------------------------
    # mouse_sc_rnase
    # ------------------------------------------------------------
    elif workflow_name == "mouse_sc_rnaseq":
        gabaergic = mmc_results["class_name"].str.contains("GABA")
        glutamatergic = mmc_results["class_name"].str.contains("Glut")
        serotonergic = mmc_results["class_name"].str.contains("Sero")
        non_neuronal = ~(gabaergic | glutamatergic | serotonergic)

        mmc_results.loc[gabaergic, "phenotype"] = "GABAergic"
        mmc_results.loc[glutamatergic, "phenotype"] = "Glutamatergic"
        mmc_results.loc[serotonergic, "phenotype"] = "Serotonergic"
        mmc_results.loc[non_neuronal, "phenotype"] = mmc_results.loc[
            non_neuronal, "subclass_name"
        ]

        mmc_results.loc[glutamatergic, "rho"] = mmc_results.loc[
            glutamatergic, "class_correlation_coefficient"
        ]
        mmc_results.loc[gabaergic, "rho"] = mmc_results.loc[
            gabaergic, "class_correlation_coefficient"
        ]
        mmc_results.loc[serotonergic, "rho"] = mmc_results.loc[
            serotonergic, "class_correlation_coefficient"
        ]
        mmc_results.loc[non_neuronal, "rho"] = mmc_results.loc[
            non_neuronal, "subclass_correlation_coefficient"
        ]

        mmc_results.loc[glutamatergic, "prob"] = mmc_results.loc[
            glutamatergic, "class_bootstrapping_probability"
        ]
        mmc_results.loc[gabaergic, "prob"] = mmc_results.loc[
            gabaergic, "class_bootstrapping_probability"
        ]
        mmc_results.loc[serotonergic, "prob"] = mmc_results.loc[
            serotonergic, "class_bootstrapping_probability"
        ]
        mmc_results.loc[non_neuronal, "prob"] = mmc_results.loc[
            non_neuronal, "subclass_bootstrapping_probability"
        ]

        mmc_results["cell_type"] = mmc_results["phenotype"]

        mmc_results.loc[mmc_results["rho"] < 0.5, "cell_type"] = "Unknown"
        mmc_results.loc[mmc_results["prob"] < 0.5, "cell_type"] = "Unknown"

        return mmc_results[
            [
                "cell_type",
                "phenotype",
                "rho",
                "prob",
                "class_name",
                "subclass_name",
                "supertype_name",
                "cluster_name",
                "cluster_correlation_coefficient",
                "cluster_bootstrapping_probability",
            ]
        ]

    # ------------------------------------------------------------
    # BASAL GANGLIA
    # ------------------------------------------------------------
    elif workflow_name == "basal_ganglia":
        class_mapper = {
            "OPC-Oligo": "OPC-Oligo",
            "Astro-Epen": "Astrocyte",
            "Vascular": "Vascular",
            "Immune": "Immune",
            "F M Glut": "Glutamatergic",
            "M Dopa": "Dopaminergic",
            "CN CGE GABA": "GABAergic",
            "F M GABA": "GABAergic",
            "CN MGE GABA": "GABAergic",
            "CN LGE GABA": "GABAergic",
            "Cx GABA": "GABAergic",
            "CN GABA-Glut": "GABAergic",
        }

        mmc_results["phenotype"] = mmc_results["Class_name"].map(class_mapper)

        gabaergic = mmc_results["Class_name"] == "GABAergic"
        mmc_results["rho"] = mmc_results["Class_correlation_coefficient"]
        mmc_results.loc[gabaergic, "rho"] = mmc_results.loc[
            gabaergic, "Neighborhood_correlation_coefficient"
        ]

        mmc_results["prob"] = mmc_results["Class_bootstrapping_probability"]
        mmc_results.loc[gabaergic, "prob"] = mmc_results.loc[
            gabaergic, "Neighborhood_bootstrapping_probability"
        ]

        mmc_results["cell_type"] = mmc_results["phenotype"]
        mmc_results.loc[mmc_results["rho"] < 0.2, "cell_type"] = "Unknown"
        mmc_results.loc[mmc_results["prob"] < 0.5, "cell_type"] = "Unknown"

        name_mapper = {
            "Neighborhood_name": "supertype_name",
            "Class_name": "class_name",
            "Subclass_name": "subclass_name",
        }
        mmc_results.rename(columns=name_mapper, inplace=True)

        return mmc_results[
            [
                "cell_id",
                "cell_type",
                "phenotype",
                "rho",
                "prob",
                "class_name",
                "subclass_name",
                "supertype_name"
            ]
        ]

    # ------------------------------------------------------------
    # UNKNOWN WORKFLOW
    # ------------------------------------------------------------
    else:
        raise ValueError(
            f"[ERROR] Source cannot be detected from workflow name: [{workflow_name}]"
        )

In [ ]:
mmc_res = summarize_mmc_results(mmc_output, "pmdbs_sc_rnaseq")
mmc_res.head()

### 4.4 Assess MMC Results

After reformatting, we evaluate the quality and distribution of the assignments.

Key aspects to assess:

- distribution of cell_type labels
- proportion of `"Unknown"` assignments (low-confidence cells)
- ranges and distributions of `rho` (correlation strength) and `prob` (assignment stability)

This helps answer:
- Are the assignments biologically reasonable for this dataset?
- Are confidence thresholds too strict or too lenient?
- Do certain cell populations show weaker mapping signals?

In [ ]:
# Set index to cell_id for easy alignment
adata = sc.read_h5ad(processed_filename)
mapmycells_df = mmc_res.set_index("cell_id")

# validate connection 
print(len(set(mapmycells_df.index).intersection(set(adata.obs_names))))

# assign directly 
for col in ["cell_type", "class_name", "subclass_name", "supertype_name", "phenotype", "rho", "prob"]: 
    adata.obs[col] = mapmycells_df[col].values

In [ ]:
adata

#### Visualize class-level annotations on UMAP

A quick sanity check: plotting the assigned `class_name` and `prob` (bootstrapping confidence) on the UMAP embedding from Part 02. Cells should cluster coherently by class, and confidence should be higher in dense, well-separated regions.

In [ ]:
adata.obsm["X_umap"] = adata.obsm.pop("_X_umap")
sc.pl.umap(adata, color=["class_name", "prob"])

#### Restore gene names and save intermediate checkpoint

We set `var_names` to human-readable gene symbols for downstream plotting (e.g., dot plots, marker expression). We also save a checkpoint of the AnnData object with MapMyCells annotations before further processing.

In [ ]:
# move gene names back to index for readability
# 2. Set var_names to gene_names
adata.var_names = adata.var["gene_name"].astype(str)

# 3. Ensure uniqueness
adata.var_names_make_unique()

# Quick check
print(adata.var.head())
print(adata.var_names[:5])

mmc_adata_output_filepath = ( mapmycells_output_dir / f"{region}_mapmycells_integrated_output.h5ad")
adata.write_h5ad(mmc_adata_output_filepath)

## 4. Prepare data for downstream analysis

Apply standard single-cell preprocessing: normalisation, log-transform, identify HVGs and clustering (Leiden). Calculate PCA is not necessary since data has been harmonised and _X_scVI is available.

In [ ]:
# only running this if script stopped in the previous block
#mapmycells_integrated = ( mapmycells_output_dir / f"{region}_mapmycells_integrated_output.h5ad")
#adata = sc.read_h5ad(mapmycells_integrated)

In [ ]:
# Check maximum values in X and layers --> raw counts will be in the thousands, log-norm is usually 0-10
print("Max value in adata.X:", adata.X.max())

if "counts" in adata.layers:
    print("Max value in counts layer:", adata.layers["counts"].max())

# Check if a raw count matrix exists elsewhere in the object
if adata.raw is not None:
    print("Max value in adata.raw.X:", adata.raw.X.max())

In [ ]:
# Normalize counts per cell
sc.pp.normalize_total(adata, target_sum=1e4)

# Log transform the data
sc.pp.log1p(adata)

# Identify highly variable genes
hvgs_res =sc.experimental.pp.highly_variable_genes(
        adata,
        n_top_genes=3000,
        batch_key="sample", #using sample not batch
        flavor="pearson_residuals",
        check_values=True,
        layer="counts",
        subset=False,
        inplace=False
)


In [ ]:
# build a kNN graph directly from scVI Latent Space (skip PCA)
sc.pp.neighbors(adata, use_rep="_X_scVI", n_neighbors=15, method = "umap")


In [ ]:
# apply Leiden clustering at resolution 1.0
leiden_keyname = "leiden_sn_scVI_1.0"
sc.tl.leiden(adata, resolution=1.0, key_added=leiden_keyname, flavor="igraph", directed=False, random_state=0)

## 6. Export data

In [ ]:
# saving h5ad
mmc_adata_output_proc_filepath = ( mapmycells_output_dir / 
                             f"{region}_mapmycells_integrated_processed_output.h5ad")
adata.write_h5ad(mmc_adata_output_proc_filepath)

# saving hvgs_res
hvgs_res_filepath = ( mapmycells_output_dir / 
                             f"{region}_hvgs_results.csv")
hvgs_res.to_csv(hvgs_res_filepath)